In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.io as pio

pio.renderers.default = "notebook"

In [1]:
from src.portfolio.returns import (
    calculate_returns,
    calculate_portfolio_returns,
    calculate_cumulative_returns,
)

from src.portfolio.metrics import (
    annualized_return,
    annualized_volatility,
    sharpe_ratio,
    max_drawdown,
    historical_var,
)

from src.visualization.charts import (
    plot_cumulative_returns,
    plot_drawdown,
    plot_returns_distribution,
    plot_correlation_heatmap,
)

In [4]:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "SPY"]

weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2])

start_date = "2020-01-01"
end_date = "2025-01-01"

In [5]:
prices = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False,
    threads=False,
)["Close"]

prices.head()

Ticker,AAPL,AMZN,GOOGL,MSFT,SPY
Date,,,,,
2020-01-02,72.333885,94.900497,67.873016,151.829575,296.888184
2020-01-03,71.630638,93.748497,67.517967,149.939011,294.640045
2020-01-06,72.201416,95.143997,69.317596,150.326569,295.764191
2020-01-07,71.861839,95.343002,69.183701,148.955933,294.932587
2020-01-08,73.017830,94.598503,69.676132,151.328568,296.504395


In [6]:
asset_returns = calculate_returns(prices)

portfolio_returns = calculate_portfolio_returns(
    asset_returns=asset_returns,
    weights=weights,
)

cumulative_returns = calculate_cumulative_returns(portfolio_returns)

asset_returns.head()

Ticker,AAPL,AMZN,GOOGL,MSFT,SPY
Date,,,,,
2020-01-03,-0.009722,-0.012139,-0.005231,-0.012452,-0.007572
2020-01-06,0.007968,0.014886,0.026654,0.002585,0.003815
2020-01-07,-0.004703,0.002092,-0.001932,-0.009118,-0.002812
2020-01-08,0.016086,-0.007809,0.007118,0.015928,0.005329
2020-01-09,0.021241,0.004799,0.010498,0.012493,0.006781


In [6]:
running_max = cumulative_returns.cummax()
drawdown = cumulative_returns / running_max - 1

drawdown.head()

Date
2020-01-03    0.000000
2020-01-06    0.000000
2020-01-07   -0.003294
2020-01-08    0.000000
2020-01-09    0.000000
dtype: float64

In [7]:
metrics = {
    "Annualized Return": annualized_return(portfolio_returns),
    "Annualized Volatility": annualized_volatility(portfolio_returns),
    "Sharpe Ratio": sharpe_ratio(portfolio_returns),
    "Max Drawdown": max_drawdown(cumulative_returns),
    "Historical VaR 95%": historical_var(portfolio_returns),
}

metrics_df = pd.DataFrame(metrics, index=["Value"]).T
metrics_df

,Value
Annualized Return,0.238545
Annualized Volatility,0.264018
Sharpe Ratio,0.903520
Max Drawdown,-0.352634
Historical VaR 95%,-0.025727


In [8]:
formatted_metrics = pd.DataFrame(
    {
        "Value": [
            f"{metrics['Annualized Return']:.2%}",
            f"{metrics['Annualized Volatility']:.2%}",
            f"{metrics['Sharpe Ratio']:.2f}",
            f"{metrics['Max Drawdown']:.2%}",
            f"{metrics['Historical VaR 95%']:.2%}",
        ]
    },
    index=[
        "Annualized Return",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Max Drawdown",
        "Historical VaR 95%",
    ],
)

formatted_metrics

,Value
Annualized Return,23.85%
Annualized Volatility,26.40%
Sharpe Ratio,0.90
Max Drawdown,-35.26%
Historical VaR 95%,-2.57%


In [9]:
plot_cumulative_returns(cumulative_returns).show()

In [10]:
plot_drawdown(drawdown).show()

In [11]:
plot_returns_distribution(portfolio_returns).show()

In [12]:
plot_correlation_heatmap(asset_returns).show()